In [12]:
import sys
sys.path.insert(0, "../src")
from path_config import set_path_mapping, get_path_mapping
import json
import os
import shutil

In [13]:
def get_glance_url(folder_path: str, filenames: list[str]) -> str:
    base = "http://localhost:3000"
    mesh_urls = []
    if len(filenames) > 1:
        for file in filenames:
            url = f"{base}/{folder_path}/{file}"
            mesh_urls.append(url)
        return f"{base}?name=[{','.join(filenames)}]&url=[{','.join(mesh_urls)}]"
    else:
        url = f"{base}/{folder_path}/{filenames[0]}"
        return f"{base}?name={filenames[0]}&url={url}"

In [14]:
# path to the folder with COBIVECO mesh results
COBIVECO_OUTPUT_FOLDER = "/mnt/data-vm2/Cobiveco/input-data"
# path to the folder with TORSO-MPP results
TORSO_PIPELINE_OUTPUT_FOLDER = "/mnt/data-vm2/TORSO-MPP/data-fitted"
# path to INPUT-DATA for Cardiac-Digital-Twin framework
#CDT_DATA_FOLDER = "/mnt/cardiac-data/meta_data/"
CDT_DATA_FOLDER = "/data/Cardiac-Digital-Twin/cardiac-data/meta_data/"
# path to the folder to save results Cardiac-Digital-Twin framework RESULTS
CDT_RESULTS_FOLDER = "/data/Cardiac-Digital-Twin/cardiac-data/meta_data/results_test/"

path_mapping_dict = {
    "cobiveco_output_path": COBIVECO_OUTPUT_FOLDER,
    "torso_output_path": TORSO_PIPELINE_OUTPUT_FOLDER,
    "data_path": CDT_DATA_FOLDER,
    "results_path": CDT_RESULTS_FOLDER
}
set_path_mapping(json.dumps(path_mapping_dict))

{'cobiveco_output_path': '/mnt/data-vm2/Cobiveco/input-data', 'torso_output_path': '/mnt/data-vm2/TORSO-MPP/data-fitted', 'data_path': '/data/Cardiac-Digital-Twin/cardiac-data/meta_data/', 'results_path': '/data/Cardiac-Digital-Twin/cardiac-data/meta_data/results_test/'}


In [17]:
#import mesh from cobiveco, scaling to coarse and fine resolution and saving coordinates csv
CASE_NAME = "sb3101"
coarse_resolution = '1500'
fine_resolution = '500'
cobiveco_um = 'mm'
cobiveco_output_coarse = os.path.join(path_mapping_dict["cobiveco_output_path"], f"{CASE_NAME}_{coarse_resolution+cobiveco_um}_resFibers")
cobiveco_output_fine = os.path.join(path_mapping_dict["cobiveco_output_path"], f"{CASE_NAME}_{fine_resolution+cobiveco_um}_resFibers")
geometric_data_dir = os.path.join(path_mapping_dict["data_path"], "geometric_data")
cdt_coarse_um = 'cm'
cdt_fine_um = 'um'
subject_geom_dir = os.path.join(geometric_data_dir, CASE_NAME)
if not os.path.exists(subject_geom_dir):
    os.makedirs(subject_geom_dir)
subject_geom_dir_meshes = os.path.join(subject_geom_dir, "cardiac_meshes_original")
if not os.path.exists(subject_geom_dir_meshes):
    os.makedirs(subject_geom_dir_meshes)

In [ ]:
from cobiveco_export_functions import scale_mesh_mantaining_point_data, save_vtk_vtu_to_csv, save_vtu_arrays_to_csv, export_electrodes_to_csv, extract_ids_from_tagged_vtp

#SCALING MESH coarse from mm to cm for CDT framework
input_filename =  f"{CASE_NAME}_{coarse_resolution+cobiveco_um}_fibers.vtu" #from mm to cm
input_path = os.path.join(cobiveco_output_coarse, input_filename)
#copy also original vtu file in mm
shutil.copy(input_path, subject_geom_dir_meshes)

out_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}_fibers.vtu" #from mm to cm
out_path = os.path.join(subject_geom_dir_meshes, out_filename)

scale_mesh_mantaining_point_data(input_path, out_path, scale=0.1) # scale mm to cm

input_filename = f"{CASE_NAME}_{coarse_resolution+cobiveco_um}.vtp" #from mm to cm
input_path = os.path.join(cobiveco_output_coarse, input_filename)
out_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}.vtp"
out_path = os.path.join(subject_geom_dir_meshes, out_filename)
scale_mesh_mantaining_point_data(input_path, out_path, scale=0.1)  # scale mm to cm 

#scaling mesh fine from mm to um for monodomain simulation 
input_filename = f"{CASE_NAME}_{fine_resolution+cobiveco_um}.vtp" # da mm a um 
input_path = os.path.join(cobiveco_output_fine, input_filename)
out_filename = f"{CASE_NAME}_{fine_resolution+cdt_fine_um}.vtp" # da mm a um
out_path = os.path.join(subject_geom_dir_meshes, out_filename)
scale_mesh_mantaining_point_data(input_path, out_path, scale=1000) # scale mm to um

input_filename = f"{CASE_NAME}_{fine_resolution+cobiveco_um}_fibers.vtk" # da mm a um
input_path = os.path.join(cobiveco_output_fine, input_filename)
out_filename = f"{CASE_NAME}_{fine_resolution+cdt_fine_um}_fibers.vtk" # da mm a um
out_path = os.path.join(subject_geom_dir_meshes, out_filename)
scale_mesh_mantaining_point_data(input_path, out_path, scale=1000)

Bounds mesh originale: BoundsTuple(x_min =  -10.0525868226031,
            x_max =  121.065026280837,
            y_min = -124.462942411429,
            y_max =   -7.69171352785344,
            z_min =  -80.7046030681628,
            z_max =   44.8908068740297)
Point data preservati: ['tv', 'tm', 'rtSin', 'rtCos', 'rt', 'ab', 'lvrv', 'aprt']
Mesh scalata salvata in: /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_1500cm_fibers.vtu
mesh scalata di un fattore: 0.1
Bounds mesh scalata: BoundsTuple(x_min =  -1.00525868226031,
            x_max =  12.106502628083701,
            y_min = -12.446294241142901,
            y_max =  -0.7691713527853441,
            z_min =  -8.07046030681628,
            z_max =   4.4890806874029705)
Bounds mesh originale: BoundsTuple(x_min =  -10.0525868226031,
            x_max =  121.065026280837,
            y_min = -124.462942411429,
            y_max =   -7.69171352785344,
            z_min =  -80.7046

In [8]:
torso_case_folder = os.path.join(path_mapping_dict["torso_output_path"], CASE_NAME)
electrodes_vtk_orig_name = 'ECG_ELECTRODES.vtk'
torso_vtk_orig_name = 'TORSO.vtk'
heartssm_aligned_reference_name = 'HEART_aligned.vtk' 

output_dir = os.path.join(subject_geom_dir, "torso_meshes")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

shutil.copy( os.path.join(torso_case_folder, electrodes_vtk_orig_name), os.path.join(output_dir, CASE_NAME+'_ECG_ELECTRODESmm.vtk'))
shutil.copy( os.path.join(torso_case_folder, torso_vtk_orig_name), os.path.join(output_dir, CASE_NAME+'_TORSOmm.vtk'))
shutil.copy( os.path.join(torso_case_folder, 'mpp', heartssm_aligned_reference_name), os.path.join(output_dir, CASE_NAME+'_HEARTssm_aligned_reference.vtk'))

csv_output = os.path.join(output_dir, CASE_NAME + '_electrode_xyz.csv')
export_electrodes_to_csv(os.path.join(output_dir,CASE_NAME+'_ECG_ELECTRODESmm.vtk'), csv_output, scale=0.1)

In [9]:
glance_meshes = [
    CASE_NAME+'_ECG_ELECTRODESmm.vtk',
    CASE_NAME+'_TORSOmm.vtk',
    CASE_NAME+'_HEARTssm_aligned_reference.vtk',
    f"{CASE_NAME}_{coarse_resolution+cobiveco_um}_fibers.vtu"

]
url = get_glance_url(output_dir,glance_meshes)
print(url)

http://localhost:3000?name=[sb3101_ECG_ELECTRODESmm.vtk,sb3101_TORSOmm.vtk,sb3101_HEARTssm_aligned_reference.vtk,sb3101_1500mm_fibers.vtu]&url=[http://localhost:3000//data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/torso_meshes/sb3101_ECG_ELECTRODESmm.vtk,http://localhost:3000//data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/torso_meshes/sb3101_TORSOmm.vtk,http://localhost:3000//data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/torso_meshes/sb3101_HEARTssm_aligned_reference.vtk,http://localhost:3000//data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/torso_meshes/sb3101_1500mm_fibers.vtu]


In [7]:
from torso_heart_registration import compute_registration_transform, apply_tranform_to_meshes
geometric_data_dir = os.path.join(path_mapping_dict["data_path"], "geometric_data")
subject_geom_dir = os.path.join(geometric_data_dir, CASE_NAME)
subject_geom_dir_meshes = os.path.join(subject_geom_dir, "cardiac_meshes_original")

input_dir_torso = os.path.join(subject_geom_dir, "torso_meshes")
input_cardiac_mesh_filename =  f"{CASE_NAME}_{coarse_resolution+cobiveco_um}_fibers.vtu"
T_final = compute_registration_transform(case_name=CASE_NAME, input_meshes_folder=subject_geom_dir_meshes, input_torso_folder=input_dir_torso, 
                               cardiac_mesh_name=input_cardiac_mesh_filename, 
                               ssm_heart_from_torso_name=CASE_NAME+'_HEARTssm_aligned_reference.vtk', 
                               voxel_size = 3)

/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:215: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  biv_surface = biv_mesh.extract_surface()
/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:220: PyVistaFutureWarning: The default value of `algorithm` for the filter
`PolyData.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  ssmheart_from_torso_surface = ssm_heart_mesh.extract_surface()


Global registration using RANSAC...
RANSAC fitness: 0.3658, RMSE: 2.45


/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:229: PyVistaDeprecationWarning: The default value of `inplace` for the filter `UnstructuredGrid.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  biv_after_ransac = biv_mesh.copy().transform(T_global_ransac)


Num Normali: 38039
normali zero: 0
norma media: 1.0000000008525005
Local registration using ICP...
registration matrix saved at: /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_icp_transform.npy
ICP fitness: 0.2549, RMSE: 0.798


In [10]:
meshes_to_reg = [
    f"{CASE_NAME}_500um_fibers.vtk",
    f"{CASE_NAME}_500um.vtp",
    f"{CASE_NAME}_1500cm_fibers.vtu",
    f"{CASE_NAME}_1500cm.vtp",
    f"{CASE_NAME}_1500mm_fibers.vtu"
]
input_folder = subject_geom_dir_meshes
output_folder = os.path.join(subject_geom_dir, "cardiac_meshes_reg")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

apply_tranform_to_meshes(input_folder=input_folder, output_folder=output_folder, transform_matrix=T_final, meshes_filenames=meshes_to_reg)

Processing /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_500um_fibers.vtk -> /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_500um_fibers.vtk


/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:283: PyVistaDeprecationWarning: The default value of `inplace` for the filter `UnstructuredGrid.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  mesh_registred = mesh.copy().transform(T)


Processing /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_500um.vtp -> /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_500um.vtp


/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:283: PyVistaDeprecationWarning: The default value of `inplace` for the filter `PolyData.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  mesh_registred = mesh.copy().transform(T)


Processing /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_1500cm_fibers.vtu -> /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_1500cm_fibers.vtu


/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:283: PyVistaDeprecationWarning: The default value of `inplace` for the filter `UnstructuredGrid.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  mesh_registred = mesh.copy().transform(T)


Processing /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_1500cm.vtp -> /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_1500cm.vtp
Processing /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_original/sb3101_1500mm_fibers.vtu -> /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_1500mm_fibers.vtu


/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:283: PyVistaDeprecationWarning: The default value of `inplace` for the filter `PolyData.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  mesh_registred = mesh.copy().transform(T)
/data/Cardiac-Digital-Twin/opencarp-simulation-ECG/torso_heart_registration.py:283: PyVistaDeprecationWarning: The default value of `inplace` for the filter `UnstructuredGrid.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  mesh_registred = mesh.copy().transform(T)


In [19]:
from cobiveco_export_functions import scale_mesh_mantaining_point_data, save_vtk_vtu_to_csv, save_vtu_arrays_to_csv, export_electrodes_to_csv, extract_ids_from_tagged_vtp

coarse_folder_tag = 'coarse'+coarse_resolution+cdt_coarse_um
fine_folder_tag = 'fine'+fine_resolution+cdt_fine_um
def_meshes_folder = os.path.join(geometric_data_dir, CASE_NAME, 'cardiac_meshes_reg')
# EXPORT CSV nodi, tetra, coordinate cobiveco e fibre per mesh coarse
vtu_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}_fibers.vtu"
save_vtk_vtu_to_csv(CASE_NAME, geometric_data_dir, def_meshes_folder, coarse_folder_tag, vtu_filename)
save_vtu_arrays_to_csv(CASE_NAME, geometric_data_dir, def_meshes_folder, coarse_folder_tag, vtu_filename)

#export CSV solo nodi e tetra per mesh fine
fine_vtk_filename = f"{CASE_NAME}_{fine_resolution+cdt_fine_um}_fibers.vtk"
save_vtk_vtu_to_csv(CASE_NAME, geometric_data_dir, def_meshes_folder, fine_folder_tag, fine_vtk_filename)
save_vtu_arrays_to_csv(CASE_NAME, geometric_data_dir, def_meshes_folder, fine_folder_tag, fine_vtk_filename)

#export nodi endo rv e lv mesh coarse
vtu_path = os.path.join(def_meshes_folder, vtu_filename)
output_dir = os.path.join(geometric_data_dir, CASE_NAME, f"{CASE_NAME}_{coarse_folder_tag}")
vtp_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}.vtp"
vtp_path = os.path.join(def_meshes_folder, vtp_filename)

TAG_LV_ENDO = 3  # Valore per Endocardio LV
TAG_RV_ENDO = 4  # Valore per Endocardio RV
TAG_ARRAY_NAME = 'class'

extract_ids_from_tagged_vtp(
    vtu_path=vtu_path,
    vtp_path=vtp_path,
    tag_array_name=TAG_ARRAY_NAME,
    target_tag_value=TAG_LV_ENDO,
    output_csv=os.path.join(output_dir, CASE_NAME + '_' + coarse_folder_tag + '_boundarynodefield' + '_lvendo' + '.csv')
)
# Genera file RV
extract_ids_from_tagged_vtp(
    vtu_path=vtu_path,
    vtp_path=vtp_path,
    tag_array_name=TAG_ARRAY_NAME,
    target_tag_value=TAG_RV_ENDO,
    output_csv=os.path.join(output_dir, CASE_NAME + '_' + coarse_folder_tag + '_boundarynodefield' + '_rvendo' + '.csv')
)

(141468, 3)
(709911, 4)
(3629336, 3)
(20756259, 4)
--- Elaborazione Tag 3 da /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_1500cm.vtp ---
  Punti superficie estratti: 9315
  Punti volume totale: 141468
  SALVATO: /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/sb3101_coarse1500cm/sb3101_coarse1500cm_boundarynodefield_lvendo.csv con 9315 nodi unici.

--- Elaborazione Tag 4 da /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_1500cm.vtp ---
  Punti superficie estratti: 10488
  Punti volume totale: 141468
  SALVATO: /data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/sb3101_coarse1500cm/sb3101_coarse1500cm_boundarynodefield_rvendo.csv con 10488 nodi unici.



In [20]:
input_filename = f"{CASE_NAME}_{fine_resolution+cobiveco_um}_fibersOpencarp.lon" # da mm a um 
input_path = os.path.join(cobiveco_output_fine, input_filename)
def_meshes_folder = os.path.join(geometric_data_dir, CASE_NAME, 'cardiac_meshes_reg')
shutil.copy(input_path, def_meshes_folder)

'/data/Cardiac-Digital-Twin/cardiac-data/meta_data/geometric_data/sb3101/cardiac_meshes_reg/sb3101_500mm_fibersOpencarp.lon'